<a href="https://colab.research.google.com/github/NikhithaReddy77/Dynamic-Data-Contract-Drift-Revenue-Impact-Profiler/blob/main/Personal_project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Detects schema drift, quantifies revenue impact, proposes fixes**

In [1]:
!pip install pydantic networkx plotly ipywidgets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 44.8 MB/s eta 0:00:00


In [2]:
from __future__ import annotations
from datetime import datetime
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field


class DriftType(str, Enum):
    COLUMN_ADDED = "column_added"
    COLUMN_REMOVED = "column_removed"
    COLUMN_RENAMED = "column_renamed"
    TYPE_CHANGED = "type_changed"
    NULLABILITY_CHANGED = "nullability_changed"


class ColumnSchema(BaseModel):
    name: str
    data_type: str
    nullable: bool = True
    sample_values: list[str] = Field(default_factory=list)


class TableSchema(BaseModel):
    database: str
    schema_name: str
    table: str
    columns: list[ColumnSchema]
    captured_at: datetime = Field(default_factory=datetime.utcnow)

    @property
    def fqn(self) -> str:
        return f"{self.database}.{self.schema_name}.{self.table}"


class DriftEvent(BaseModel):
    table_fqn: str
    drift_type: DriftType
    old_column: Optional[str] = None
    new_column: Optional[str] = None
    old_type: Optional[str] = None
    new_type: Optional[str] = None
    confidence: float = 1.0
    detected_at: datetime = Field(default_factory=datetime.utcnow)


class DownstreamAsset(BaseModel):
    asset_id: str
    asset_type: str
    name: str
    revenue_per_hour: float
    column_criticality: dict[str, float] = Field(default_factory=dict)


class RevenueImpact(BaseModel):
    drift_event: DriftEvent
    affected_assets: list[str]
    estimated_dollars_per_hour: float
    confidence_low: float
    confidence_high: float
    minutes_since_detected: float
    estimated_dollars_exposed_so_far: float


class AdapterSuggestion(BaseModel):
    drift_event: DriftEvent
    suggested_sql: str
    mapping_confidence: float
    rationale: str
    requires_human_approval: bool = True
    approved: bool = False

In [3]:
from difflib import SequenceMatcher

RENAME_CONFIDENCE_THRESHOLD = 0.45


def _name_similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


def _sample_overlap(a: ColumnSchema, b: ColumnSchema) -> float:
    if not a.sample_values or not b.sample_values:
        return 0.0
    set_a, set_b = set(a.sample_values), set(b.sample_values)
    return len(set_a & set_b) / max(len(set_a | set_b), 1)


def _rename_score(removed: ColumnSchema, added: ColumnSchema) -> float:
    name_sim = _name_similarity(removed.name, added.name)
    type_match = 1.0 if removed.data_type == added.data_type else 0.0
    value_overlap = _sample_overlap(removed, added)
    return (0.5 * name_sim) + (0.25 * type_match) + (0.25 * value_overlap)


def diff_schemas(old: TableSchema, new: TableSchema) -> list[DriftEvent]:
    if old.fqn != new.fqn:
        raise ValueError(f"Cannot diff schemas for different tables: {old.fqn} vs {new.fqn}")

    old_cols = {c.name: c for c in old.columns}
    new_cols = {c.name: c for c in new.columns}

    removed_names = set(old_cols) - set(new_cols)
    added_names = set(new_cols) - set(old_cols)
    common_names = set(old_cols) & set(new_cols)

    events: list[DriftEvent] = []

    for name in common_names:
        old_c, new_c = old_cols[name], new_cols[name]
        if old_c.data_type != new_c.data_type:
            events.append(DriftEvent(
                table_fqn=new.fqn, drift_type=DriftType.TYPE_CHANGED,
                old_column=name, new_column=name,
                old_type=old_c.data_type, new_type=new_c.data_type, confidence=1.0,
            ))
        if old_c.nullable != new_c.nullable:
            events.append(DriftEvent(
                table_fqn=new.fqn, drift_type=DriftType.NULLABILITY_CHANGED,
                old_column=name, new_column=name, confidence=1.0,
            ))

    unmatched_removed = set(removed_names)
    unmatched_added = set(added_names)

    candidates = []
    for r_name in removed_names:
        for a_name in added_names:
            score = _rename_score(old_cols[r_name], new_cols[a_name])
            if score >= RENAME_CONFIDENCE_THRESHOLD:
                candidates.append((score, r_name, a_name))
    candidates.sort(reverse=True, key=lambda x: x[0])

    for score, r_name, a_name in candidates:
        if r_name in unmatched_removed and a_name in unmatched_added:
            events.append(DriftEvent(
                table_fqn=new.fqn, drift_type=DriftType.COLUMN_RENAMED,
                old_column=r_name, new_column=a_name,
                old_type=old_cols[r_name].data_type, new_type=new_cols[a_name].data_type,
                confidence=round(score, 2),
            ))
            unmatched_removed.discard(r_name)
            unmatched_added.discard(a_name)

    for name in unmatched_removed:
        events.append(DriftEvent(table_fqn=new.fqn, drift_type=DriftType.COLUMN_REMOVED,
                                  old_column=name, confidence=1.0))
    for name in unmatched_added:
        events.append(DriftEvent(table_fqn=new.fqn, drift_type=DriftType.COLUMN_ADDED,
                                  new_column=name, confidence=1.0))

    return events

In [4]:
import networkx as nx


class LineageGraph:
    def __init__(self) -> None:
        self.graph = nx.DiGraph()

    def add_table(self, table_fqn: str) -> None:
        self.graph.add_node(table_fqn, kind="table")

    def add_asset(self, asset: DownstreamAsset) -> None:
        self.graph.add_node(asset.asset_id, kind=asset.asset_type, asset=asset)

    def link(self, table_fqn: str, asset_id: str, columns_used: list[str]) -> None:
        if table_fqn not in self.graph:
            self.add_table(table_fqn)
        self.graph.add_edge(table_fqn, asset_id, columns=set(columns_used))

    def downstream_assets_for_column(self, table_fqn: str, column: str) -> list[DownstreamAsset]:
        if table_fqn not in self.graph:
            return []
        affected: list[DownstreamAsset] = []
        visited = set()

        def _walk(node: str):
            for _, neighbor, data in self.graph.out_edges(node, data=True):
                cols = data.get("columns")
                touches = (cols is None) or (column in cols)
                if not touches or neighbor in visited:
                    continue
                visited.add(neighbor)
                node_data = self.graph.nodes[neighbor]
                if "asset" in node_data:
                    affected.append(node_data["asset"])
                _walk(neighbor)

        _walk(table_fqn)
        return affected

In [5]:
from datetime import datetime, timezone

BASE_UNCERTAINTY_PCT = 0.15


def calculate_impact(drift: DriftEvent, lineage: LineageGraph, now: datetime | None = None) -> RevenueImpact:
    now = now or datetime.now(timezone.utc)

    # Lineage edges are recorded against whatever column name was live when
    # the graph was built. For renames, that's the OLD name -- check both
    # old and new names and merge, deduplicating by asset_id.
    candidate_columns = [c for c in (drift.old_column, drift.new_column) if c]
    seen_asset_ids: set[str] = set()
    matched_assets = []
    for col in candidate_columns:
        for asset in lineage.downstream_assets_for_column(drift.table_fqn, col):
            if asset.asset_id not in seen_asset_ids:
                seen_asset_ids.add(asset.asset_id)
                matched_assets.append((asset, col))

    point_estimate = 0.0
    for asset, matched_column in matched_assets:
        criticality = asset.column_criticality.get(matched_column, 0.5)
        point_estimate += asset.revenue_per_hour * criticality

    assets = [a for a, _ in matched_assets]

    uncertainty_pct = BASE_UNCERTAINTY_PCT + (1 - drift.confidence) * 0.35
    low = point_estimate * (1 - uncertainty_pct) * drift.confidence
    high = point_estimate * (1 + uncertainty_pct)

    minutes_elapsed = max((now - drift.detected_at.replace(tzinfo=timezone.utc)
                            if drift.detected_at.tzinfo is None
                            else now - drift.detected_at).total_seconds() / 60, 0)
    dollars_exposed_so_far = point_estimate * (minutes_elapsed / 60)

    return RevenueImpact(
        drift_event=drift,
        affected_assets=[a.asset_id for a in assets],
        estimated_dollars_per_hour=round(point_estimate, 2),
        confidence_low=round(max(low, 0), 2),
        confidence_high=round(high, 2),
        minutes_since_detected=round(minutes_elapsed, 1),
        estimated_dollars_exposed_so_far=round(dollars_exposed_so_far, 2),
    )

In [6]:
def generate_adapter(drift: DriftEvent) -> AdapterSuggestion | None:
    if drift.drift_type != DriftType.COLUMN_RENAMED:
        return None

    table = drift.table_fqn
    old_col, new_col = drift.old_column, drift.new_column

    sql = (
        f"-- Auto-generated compatibility view. Review before applying.\n"
        f"CREATE OR REPLACE VIEW {table}_COMPAT AS\n"
        f"SELECT *,\n"
        f"       {new_col} AS {old_col}  -- restores old column name for existing consumers\n"
        f"FROM {table};"
    )

    rationale = (
        f"Schema differ inferred `{old_col}` was renamed to `{new_col}` "
        f"(confidence {drift.confidence:.0%}, based on name similarity, "
        f"type match, and sample-value overlap). This view lets existing "
        f"queries/dashboards referencing `{old_col}` keep working while "
        f"downstream owners migrate to `{new_col}` on their own schedule."
    )

    return AdapterSuggestion(
        drift_event=drift, suggested_sql=sql, mapping_confidence=drift.confidence,
        rationale=rationale, requires_human_approval=True, approved=False,
    )

In [7]:
def get_schema_before() -> TableSchema:
    return TableSchema(
        database="ANALYTICS", schema_name="PUBLIC", table="CHECKOUT_EVENTS",
        columns=[
            ColumnSchema(name="event_id", data_type="VARCHAR", nullable=False,
                         sample_values=["evt_001", "evt_002", "evt_003"]),
            ColumnSchema(name="user_id", data_type="VARCHAR", nullable=False,
                         sample_values=["usr_9f2", "usr_1a4", "usr_7c3"]),
            ColumnSchema(name="discount_pct", data_type="FLOAT", nullable=True,
                         sample_values=["0.10", "0.0", "0.25"]),
            ColumnSchema(name="order_total", data_type="FLOAT", nullable=False,
                         sample_values=["49.99", "120.00", "18.50"]),
            ColumnSchema(name="internal_notes", data_type="VARCHAR", nullable=True,
                         sample_values=["", "vip customer", ""]),
        ],
    )


def get_schema_after() -> TableSchema:
    return TableSchema(
        database="ANALYTICS", schema_name="PUBLIC", table="CHECKOUT_EVENTS",
        columns=[
            ColumnSchema(name="event_id", data_type="VARCHAR", nullable=False,
                         sample_values=["evt_001", "evt_002", "evt_003"]),
            ColumnSchema(name="customer_id", data_type="VARCHAR", nullable=False,
                         sample_values=["usr_9f2", "usr_1a4", "usr_7c3"]),
            ColumnSchema(name="discount_pct", data_type="FLOAT", nullable=True,
                         sample_values=["0.10", "0.0", "0.25"]),
            ColumnSchema(name="order_total", data_type="FLOAT", nullable=False,
                         sample_values=["49.99", "120.00", "18.50"]),
            ColumnSchema(name="internal_notes", data_type="VARCHAR", nullable=True,
                         sample_values=["", "vip customer", ""]),
        ],
    )


def get_lineage_graph() -> LineageGraph:
    lineage = LineageGraph()
    table = get_schema_before().fqn

    checkout_dashboard = DownstreamAsset(
        asset_id="dash_checkout_revenue", asset_type="dashboard",
        name="Checkout Revenue (Exec Dashboard)", revenue_per_hour=42_000.0,
        column_criticality={"user_id": 0.9, "discount_pct": 0.8, "order_total": 1.0, "internal_notes": 0.05},
    )
    fraud_model = DownstreamAsset(
        asset_id="ml_fraud_scoring", asset_type="ml_model",
        name="Real-time Fraud Scoring Model", revenue_per_hour=15_500.0,
        column_criticality={"user_id": 0.95, "discount_pct": 0.2, "order_total": 0.6, "internal_notes": 0.0},
    )

    lineage.add_asset(checkout_dashboard)
    lineage.add_asset(fraud_model)
    lineage.link(table, checkout_dashboard.asset_id,
                 columns_used=["user_id", "discount_pct", "order_total", "internal_notes"])
    lineage.link(table, fraud_model.asset_id, columns_used=["user_id", "discount_pct", "order_total"])

    return lineage

In [8]:
before = get_schema_before()
after = get_schema_after()
lineage = get_lineage_graph()

print(f"Diffing schema for {before.fqn} ...\n")
events = diff_schemas(before, after)

results = []
for event in events:
    print(f"DRIFT DETECTED: {event.drift_type.value} ({event.old_column} -> {event.new_column}) "
          f"confidence={event.confidence:.0%}")
    impact = calculate_impact(event, lineage)
    print(f"  Estimated exposure: ${impact.confidence_low:,.0f}-${impact.confidence_high:,.0f}/hr "
          f"(point estimate ${impact.estimated_dollars_per_hour:,.0f}/hr)")
    print(f"  Affected assets: {impact.affected_assets}\n")
    results.append((event, impact))

    adapter = generate_adapter(event)
    if adapter:
        print("  Proposed adapter (requires human approval):")
        print("  " + adapter.suggested_sql.replace("\n", "\n  "))
        print(f"  Rationale: {adapter.rationale}\n")
    print("-" * 70)

Diffing schema for ANALYTICS.PUBLIC.CHECKOUT_EVENTS ...

DRIFT DETECTED: column_renamed (user_id -> customer_id) confidence=89%
  Estimated exposure: $37,935-$62,426/hr (point estimate $52,525/hr)
  Affected assets: ['dash_checkout_revenue', 'ml_fraud_scoring']

  Proposed adapter (requires human approval):
  -- Auto-generated compatibility view. Review before applying.
  CREATE OR REPLACE VIEW ANALYTICS.PUBLIC.CHECKOUT_EVENTS_COMPAT AS
  SELECT *,
         customer_id AS user_id  -- restores old column name for existing consumers
  FROM ANALYTICS.PUBLIC.CHECKOUT_EVENTS;
  Rationale: Schema differ inferred `user_id` was renamed to `customer_id` (confidence 89%, based on name similarity, type match, and sample-value overlap). This view lets existing queries/dashboards referencing `user_id` keep working while downstream owners migrate to `customer_id` on their own schedule.

----------------------------------------------------------------------


In [30]:
import plotly.graph_objects as go

# Build results list from the real Snowflake run
real_results = []
for event in events:
    impact = calculate_impact(event, real_lineage)
    real_results.append((event, impact))

labels = [f"{e.old_column} -> {e.new_column}" if e.new_column and e.old_column else (e.new_column or e.old_column)
          for e, _ in real_results]
low = [i.confidence_low for _, i in real_results]
high = [i.confidence_high for _, i in real_results]
point = [i.estimated_dollars_per_hour for _, i in real_results]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=labels, y=point,
    error_y=dict(type="data", symmetric=False,
                 array=[h - p for h, p in zip(high, point)],
                 arrayminus=[p - l for p, l in zip(point, low)]),
    marker_color="#1D9E75", name="Estimated $/hour exposure"
))
fig.update_layout(title="Revenue exposure by drift event (live Snowflake data)",
                   yaxis_title="$ / hour", template="plotly_white", showlegend=False)
fig.show()

In [31]:
import plotly.express as px
import pandas as pd

asset_nodes = [n for n, d in real_lineage.graph.nodes(data=True) if "asset" in d]
rows = []
for node_id in asset_nodes:
    asset = real_lineage.graph.nodes[node_id]["asset"]
    for col, crit in asset.column_criticality.items():
        rows.append({"asset": asset.name, "column": col, "criticality": crit})

df = pd.DataFrame(rows)
pivot = df.pivot(index="asset", columns="column", values="criticality")
fig = px.imshow(pivot, text_auto=".0%", color_continuous_scale="Oranges",
                 title="Column criticality by downstream asset (live data)")
fig.show()

In [13]:
import ipywidgets as widgets
from IPython.display import display
from datetime import timedelta

rename_event = next(e for e in events if e.drift_type.value == "column_renamed")

output = widgets.Output()
slider = widgets.IntSlider(value=0, min=0, max=180, step=5, description="Minutes elapsed:")

def on_change(change):
    output.clear_output()
    with output:
        fake_detected_at = datetime.now(timezone.utc) - timedelta(minutes=change["new"])
        e = rename_event.model_copy(update={"detected_at": fake_detected_at})
        impact = calculate_impact(e, lineage)
        print(f"After {change['new']} minutes undetected:")
        print(f"  Exposure rate: ${impact.estimated_dollars_per_hour:,.0f}/hr "
              f"(range ${impact.confidence_low:,.0f}-${impact.confidence_high:,.0f})")
        print(f"  Total exposed so far: ${impact.estimated_dollars_exposed_so_far:,.2f}")

slider.observe(on_change, names="value")
display(slider, output)
on_change({"new": slider.value})

IntSlider(value=0, description='Minutes elapsed:', max=180, step=5)

Output()

In [14]:
!pip install snowflake-connector-python -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 6.2 MB/s eta 0:00:00


In [18]:
from google.colab import userdata

SNOWFLAKE_ACCOUNT = userdata.get('SNOWFLAKE_ACCOUNT')
SNOWFLAKE_USER = userdata.get('SNOWFLAKE_USER')
SNOWFLAKE_PASSWORD = userdata.get('SNOWFLAKE_PASSWORD')
SNOWFLAKE_WAREHOUSE = userdata.get('SNOWFLAKE_WAREHOUSE')
SNOWFLAKE_ROLE = userdata.get('SNOWFLAKE_ROLE')

print("Credentials loaded (not printed).")

Credentials loaded (not printed).


In [21]:
import snowflake.connector

def fetch_table_schema(database: str, schema: str, table: str) -> TableSchema:
    """
    Fetches a live schema snapshot from Snowflake's INFORMATION_SCHEMA.
    """
    conn = snowflake.connector.connect(
        account=SNOWFLAKE_ACCOUNT,
        user=SNOWFLAKE_USER,
        password=SNOWFLAKE_PASSWORD,
        warehouse=SNOWFLAKE_WAREHOUSE,
        role=SNOWFLAKE_ROLE,
        database=database,   # <-- sets database context on connect
        schema=schema,        # <-- sets schema context on connect
    )
    try:
        cursor = conn.cursor()
        cursor.execute(
            """
            SELECT column_name, data_type, is_nullable
            FROM information_schema.columns
            WHERE table_catalog = %s AND table_schema = %s AND table_name = %s
            ORDER BY ordinal_position
            """,
            (database, schema, table),
        )
        columns = [
            ColumnSchema(name=row[0], data_type=row[1], nullable=(row[2] == "YES"))
            for row in cursor.fetchall()
        ]
        if not columns:
            raise ValueError(f"No columns found for {database}.{schema}.{table} — check the "
                              f"names and that your role has SELECT grants on this table.")
        return TableSchema(database=database, schema_name=schema, table=table, columns=columns)
    finally:
        conn.close()

In [22]:
live_schema = fetch_table_schema("SNOWFLAKE_SAMPLE_DATA", "TPCH_SF1", "CUSTOMER")
print(f"Connected. Found {len(live_schema.columns)} columns in {live_schema.fqn}:")
for c in live_schema.columns:
    print(f"  {c.name}: {c.data_type} (nullable={c.nullable})")

Connected. Found 8 columns in SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER:
  C_CUSTKEY: NUMBER (nullable=False)
  C_NAME: TEXT (nullable=False)
  C_ADDRESS: TEXT (nullable=False)
  C_NATIONKEY: NUMBER (nullable=False)
  C_PHONE: TEXT (nullable=False)
  C_ACCTBAL: NUMBER (nullable=False)
  C_MKTSEGMENT: TEXT (nullable=True)
  C_COMMENT: TEXT (nullable=True)


In [24]:
import json

def save_snapshot(schema: TableSchema, path: str = "snapshot.json"):
    with open(path, "w") as f:
        f.write(schema.model_dump_json())
    print(f"Snapshot saved to {path}")

def load_snapshot(path: str = "snapshot.json") -> TableSchema:
    with open(path) as f:
        return TableSchema.model_validate_json(f.read())

In [27]:
before_copy = fetch_table_schema("MY_TEST_DB", "PUBLIC", "CUSTOMER")
save_snapshot(before_copy, "customer_before.json")
print("Baseline saved from MY_TEST_DB.PUBLIC.CUSTOMER")

Snapshot saved to customer_before.json
Baseline saved from MY_TEST_DB.PUBLIC.CUSTOMER


In [28]:
old_schema = load_snapshot("customer_before.json")
new_schema = fetch_table_schema("MY_TEST_DB", "PUBLIC", "CUSTOMER")

events = diff_schemas(old_schema, new_schema)
for event in events:
    print(f"DRIFT: {event.drift_type.value} ({event.old_column} -> {event.new_column}) "
          f"confidence={event.confidence:.0%}")

DRIFT: column_renamed (C_NAME -> CUSTOMER_FULL_NAME) confidence=50%


In [29]:
def get_real_lineage_graph():
    lineage = LineageGraph()
    table = "MY_TEST_DB.PUBLIC.CUSTOMER"
    dashboard = DownstreamAsset(
        asset_id="dash_customer_360", asset_type="dashboard",
        name="Customer 360 Dashboard", revenue_per_hour=18_000.0,
        column_criticality={"C_NAME": 0.85, "C_ACCTBAL": 0.9},
    )
    lineage.add_asset(dashboard)
    lineage.link(table, dashboard.asset_id, columns_used=["C_NAME", "C_ACCTBAL"])
    return lineage

real_lineage = get_real_lineage_graph()

for event in events:
    impact = calculate_impact(event, real_lineage)
    print(f"DRIFT: {event.drift_type.value} ({event.old_column} -> {event.new_column})")
    print(f"  Exposure: ${impact.confidence_low:,.0f}-${impact.confidence_high:,.0f}/hr "
          f"(point estimate ${impact.estimated_dollars_per_hour:,.0f}/hr)")
    print(f"  Affected assets: {impact.affected_assets}")

    adapter = generate_adapter(event)
    if adapter:
        print("  Proposed adapter:")
        print("  " + adapter.suggested_sql.replace("\n", "\n  "))
    print("-" * 60)

DRIFT: column_renamed (C_NAME -> CUSTOMER_FULL_NAME)
  Exposure: $5,164-$20,272/hr (point estimate $15,300/hr)
  Affected assets: ['dash_customer_360']
  Proposed adapter:
  -- Auto-generated compatibility view. Review before applying.
  CREATE OR REPLACE VIEW MY_TEST_DB.PUBLIC.CUSTOMER_COMPAT AS
  SELECT *,
         CUSTOMER_FULL_NAME AS C_NAME  -- restores old column name for existing consumers
  FROM MY_TEST_DB.PUBLIC.CUSTOMER;
------------------------------------------------------------
